
# 2D demo : Passive cavitation imaging(PyCav)
This notebook demonstrate the use of `pycav` for passive acoustic mapping (DAS, RCB, FB, et CMF-spTV).



In [ ]:
!pip install git+https://github.com/celest-la/PyCav.git

In [ ]:
!git clone https://github.com/celest-la/PyCav.git

In [ ]:
import torch
import gc
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

# GPU optimization
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_default_dtype(torch.float32)

# Material verification
if torch.cuda.is_available():
    device = "cuda"
    gpu_name = torch.cuda.get_device_name(0)
else:
    device = "cpu"

# PyCav Imports
from pycav.core.probe import Probe
from pycav.core.grid import Grid
from pycav.solvers.beamformers import DAS, RCB, FB
from pycav.simulation.simulation import getPositions, StableSimulator, BBagSimulator
from pycav.utils.io import plot_result_dB, Timer
from pycav.solvers.preproc import compute_csm

In [ ]:
print("Probe and Grid configuration...")
probe = Probe.from_linear(n_el=128, pitch=0.3e-3, device=device)
grid = Grid.from_limits(
    x_lim=(-0.01, 0.01),
    z_lim=(0.06, 0.08),
    step=0.2e-3,
    device=device
)

positions, center, x_size, z_size, angle = getPositions.random_ellipse(
    x_position_lim=-0.004,
    z_position_lim=0.07,
    x_size_lim=2e-3,
    z_size_lim=2e-3,
    angle=0,
    density=100,
    device=device
)

fhifu = 1e6
c0 = 1480

print("RF signal simulation..")
sim_bbag = BBagSimulator(probe, positions, fhifu=fhifu, device=device, c0=c0)
rf_filtered = sim_bbag.simulate(t_acq=5e-4, n_cycles=200, noise_level=0.001)
print("RF ready")
# === GESTION MÉMOIRE CRITIQUE POUR COLAB ===
del sim_bbag
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

csm, axf = compute_csm(rf_filtered, probe, K=130, overlap=0.9, f=[5.5e6])
freqs = axf

In [ ]:
# Initialisation des solveurs
das = DAS(probe, grid, c=c0)
rcb = RCB(probe, grid, c=c0)
fb = FB(probe, grid, c=c0)

print("--- 2D reconstruction in process ---")

with Timer("DAS", device=device):
    img_das = das.solve(csm, freqs, average=True)

with Timer("RCB", device=device):
    img_rcb = rcb.solve(csm, freqs, epsilon=5, average=True)

with Timer("FB", device=device):
    img_fb = fb.solve(csm, freqs, r=0, average=True)


In [ ]:
plt.figure(figsize=(20, 5))

plt.subplot(1, 3, 1)
plot_result_dB(img_das, grid, "DAS", dynamic_range=20)

plt.subplot(1, 3, 2)
plot_result_dB(img_rcb, grid, "RCB", dynamic_range=20)

plt.subplot(1, 3, 3)
plot_result_dB(img_fb, grid, "FB (PISA)", dynamic_range=20)

plt.show()